# 2 · DDI Atomic Claim Extraction

Splits each AI-written response (`hypothesis` in `data/holistic_*.csv`) into atomic claims: one clause per drug pair that states an effect or advice. Each atom gets its own NLI label. The clause text (`hypothesis`) is what the NLI model reads; subject / relation / object are kept as metadata.

In [1]:
import re
import os
import json
import ast
from collections import Counter, defaultdict

import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans

## Settings

In [2]:
NEUTRAL_SAME_PAIR = True

# "X increases A and decreases B" -> two atoms: "X increases A" and "X decreases B".
SPLIT_CONJUNCTS = True

# Keep context phrases such as "in patients with renal impairment" or "if renal function is reduced".
# A condition cannot be told apart from filler ("in patients") reliably, so all of them are kept. They are cut
# only when the clause would otherwise be longer than MAX_LEN; those atoms get context_trimmed = True.
KEEP_CONTEXT_PHRASES = True

MIN_LEN, MAX_LEN = 4, 30        # atom length in tokens
FALLBACK_MAX_LEN = 40           # limit when a cut-off clause is repaired with the whole clause
LABELS = ("entailment", "contradiction", "neutral")

## Load the biomedical parser

In [3]:
# pip install scispacy
# pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [4]:
# Set True ONLY for a quick smoke test. The thesis pipeline is specified as scispaCy, so a silent fallback
# to en_core_web_sm changes POS/dependency/lemma output without telling you.
ALLOW_FALLBACK = False

def load_biomedical_parser():
    try:
        nlp = spacy.load("en_core_sci_sm")
        print("Loaded scispaCy en_core_sci_sm (biomedical parser).")
        return nlp
    except OSError:
        if not ALLOW_FALLBACK:
            raise RuntimeError(
                "en_core_sci_sm is not installed. Run the pip install lines in the cell above "
                "(or set ALLOW_FALLBACK = True to knowingly use en_core_web_sm)."
            )
        print("[WARNING] en_core_sci_sm missing -> using en_core_web_sm (NOT scispaCy).")
        return spacy.load("en_core_web_sm")


nlp = load_biomedical_parser()
print("pipeline:", nlp.pipe_names)

Loaded scispaCy en_core_sci_sm (biomedical parser).
pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


c:\Users\kynes\thesiskernel\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


## Drug vocabulary

In [5]:
MANIFESTS = [f"{split}_raw_manifest.csv" for split in ("train", "val", "test")]

found = [p for p in MANIFESTS if os.path.exists(p)]
if not found:
    raise FileNotFoundError(f"None of {MANIFESTS} found: run 0_ExtractData.ipynb first.")
if len(found) < len(MANIFESTS):
    print(f"[WARNING] missing {sorted(set(MANIFESTS) - set(found))}: the drug vocabulary is built from {found} only.")

rows = pd.concat([pd.read_csv(p, usecols=["e1_text", "e2_text", "premise"]) for p in found]).dropna().astype(str)
drug_names = {" ".join(n.lower().split()) for n in pd.concat([rows["e1_text"], rows["e2_text"]])}
drug_names = {n for n in drug_names if len(n) > 2 and not n.isdigit()}


# Corpus names that are mostly NOT a drug where they occur ("lead" in "may lead to", the enzyme "CYP3A4",
# "antihypertensive" in "antihypertensive effect"): a name the annotators leave unmarked in most of the corpus
# sentences containing it is not looked for as a drug mention. Same rule as notebook 1's ORDINARY_NAMES.
ORDINARY_MIN_SEEN, ORDINARY_MAX_RATE = 3, 0.5
# A name directly followed by one of these describes a kind of effect, not a drug (same list as notebook 1).
DESCRIPTOR_NOUNS = {"effect", "effects", "action", "actions", "activity", "activities", "property", "properties",
                    "potency", "efficacy", "response", "responses"}


def _norm(text):
    return " ".join(re.sub(r"[^a-z0-9]+", " ", str(text).lower()).split())


def ordinary_names(rows, names):
    index = defaultdict(list)
    for n in {_norm(n) for n in names}:
        if n:
            index[n.split()[0]].append(tuple(n.split()))
    for cands in index.values():
        cands.sort(key=len, reverse=True)
    own = defaultdict(set)
    for a, b, premise in zip(rows["e1_text"], rows["e2_text"], rows["premise"]):
        own[re.sub(r"^\[\w+\]\s*", "", premise).strip()].update((_norm(a), _norm(b)))
    seen, marked = Counter(), Counter()
    for body, drugs in own.items():
        words, i = _norm(body).split(), 0
        while i < len(words):
            for cand in index.get(words[i], ()):
                if tuple(words[i:i + len(cand)]) == cand:
                    name = " ".join(cand)
                    if i + len(cand) >= len(words) or words[i + len(cand)] not in DESCRIPTOR_NOUNS:
                        seen[name] += 1
                        marked[name] += name in drugs
                    i += len(cand) - 1
                    break
            i += 1
    return {n for n, k in seen.items() if k >= ORDINARY_MIN_SEEN and marked[n] / k < ORDINARY_MAX_RATE}


ORDINARY = ordinary_names(rows, drug_names)

# Corpus "drugs" that are also ordinary words or measures ("lead" in "may lead to", "serum potassium", "with food").
# They are only accepted as a drug when they ARE the labelled entity of the row (see entity_spans below).
GENERIC_STOP = {
    "lead","sodium","acid","agent","agents","drug","drugs","combination","type","d","n","vitamin","iron",
    "calcium","thyroid","peripheral","urinary","antibiotic","antibiotics","therapy","treatment","levels",
    "inhibitor","inhibitors","blocker","blockers","antagonist","antagonists","hydrochloride","sulfate",
    "oral","water","salt","alcohol","caffeine","folic","ethinyl","gel","cream","tablet","tablets",
    "potassium","magnesium","glucose","zinc","oxygen","food","foods","phosphate","chloride",
}

# lower-cased set of every drug word; relation extraction uses it so a drug name can never become the "relation".
# (Effect words inside group names, e.g. "drugs that inhibit CYP3A4", are taken out again after the lexicon.)
VOCAB = set()
for _n in drug_names:
    VOCAB.add(_n)
    VOCAB.update(w for w in _n.split() if len(w) > 2)
VOCAB -= GENERIC_STOP

global_patterns = [nlp.make_doc(t) for t in drug_names if t not in GENERIC_STOP and _norm(t) not in ORDINARY]
global_matcher  = PhraseMatcher(nlp.vocab, attr="LOWER")
global_matcher.add("CORPUS_DRUGS", global_patterns)

print(f"Drug vocabulary: {len(drug_names):,} names annotated in {', '.join(found)}; "
      f"{len(ORDINARY)} mostly ordinary words are only taken as the labelled drugs ({', '.join(sorted(ORDINARY)[:8])}, ...)")

Drug vocabulary: 2,415 names annotated in train_raw_manifest.csv, val_raw_manifest.csv, test_raw_manifest.csv; 43 mostly ordinary words are only taken as the labelled drugs (adrenergic, alkalinizing agents, analgesics, anticholinergic, anticholinesterase, antiretroviral, azoles, bismuth, ...)


## Atom lexicon

An atom must state an **effect or advice**, not just that the drugs were given together. The lexicon below decides what counts. Words that were added after reading generated hypotheses are kept in separate `*_ADDED` sets, so the audit at the end can count how many atoms depend on them.

In [ ]:
EFFECT_BASE = {
 "increase","decrease","reduce","enhance","potentiate","inhibit","raise","lower","prolong","elevate","diminish",
 "attenuate","augment","boost","amplify","suppress","accelerate","slow","impair","induce","block","antagonize",
 "intensify","strengthen","weaken","limit","alter","modify","affect","cause","exacerbate","worsen","improve",
 "extend","shorten","abolish","reverse","counteract","mask","lengthen","heighten","displace","metabolize",
 "interfere","interact","compete","bind","stimulate","precipitate","trigger","promote","mediate","delay",
 "hasten","oppose","offset","neutralize","produce","result","lead","change","clear","excrete","absorb",
 "reduction","inhibition","elevation","potentiation","enhancement","induction","interaction","rise","decline",
 "displacement","competition","interference","antagonism","synergism","sensitize","predispose","contribute",
 "minimize","maximize","depress","dampen","hinder","facilitate","influence","speed","prevent","disrupt",
}
ADVISE_BASE = {"avoid","contraindicate","contraindication","contraindicated","advisable","recommend","prohibit",
               "discourage","warn","caution","monitor","adjust","require","necessitate","dangerous","unsafe"}
NO_CHANGE_BASE = {"unchanged","unaffected","unaltered","toxicity","link","associate","additive","synergistic",
                  "disruption","drop","fail","lack","ineffective","opposed"}

# Added after reading generated hypotheses. If any of these was found by reading TEST hypotheses, that is tuning
# on test data: re-derive the additions from train/val only, or disclose it in the thesis.
EFFECT_ADDED = {"cut", "halve"}                                     # "should be cut in half"
ADVISE_ADDED = {"mandate", "advise", "encourage", "support", "endorse", "urge", "instruct", "need"}
               # "mandates a reduced amount", "advised to mix", "should be encouraged",
               # "clinical recommendations support", "needs an increase"
NO_CHANGE_ADDED = {"difference", "same", "maintain", "remain", "stay", "constant", "equivalent", "comparable",
                   "finding", "target"}
               # "shows no difference", "stays the same", "maintains the profile", "known finding for",
               # "primarily target"

EFFECT        = EFFECT_BASE | EFFECT_ADDED
ADVISE        = ADVISE_BASE | ADVISE_ADDED
NO_CHANGE_ETC = NO_CHANGE_BASE | NO_CHANGE_ADDED
STRONG        = EFFECT | ADVISE | NO_CHANGE_ETC
LEXICON_ADDED = EFFECT_ADDED | ADVISE_ADDED | NO_CHANGE_ADDED

LIGHT = {"lead", "result", "contribute", "cause", "produce", "predispose", "link", "associate"}

COADMIN_VERBS = {"combine", "coadminister", "co-administer", "administer", "take", "give", "use", "prescribe", "mix"}
ADVICE_MODALS = {"should", "must", "shall", "ought"}

print(f"Lexicon: {len(STRONG)} words, of which {len(LEXICON_ADDED)} were added after reading generated text.")

# Words of a corpus group name that are effect / advice words ("inhibit" in "drugs that inhibit CYP3A4",
# "lowering" in "lipid lowering agents") stay usable as the relation. Whole drug names stay in VOCAB.
def _forms(w):
    return {w, w[:-1], w[:-2], w[:-3], w[:-3] + "e", w[:-2] + "e"} if len(w) > 4 else {w}
VOCAB = {w for w in VOCAB if w in drug_names or not (_forms(w) & STRONG)}

Lexicon: 141 words, of which 20 were added after reading generated text.


## Extraction helpers

In [ ]:
CONTEXT_PREPS   = {"during","for","in","after","before","throughout","within","among","at","under","upon","since","over"}
CONDITION_MARKS = {"if","unless","when","whenever","while","once","until","provided"}
KEEP_DEPS = {"nsubj","nsubjpass","csubj","csubjpass","dobj","attr","acomp","xcomp","ccomp","agent","dative","oprd",
             "neg","aux","auxpass","prt","advmod","npadvmod","expl","mark","prep","advcl","cc","conj","relcl","acl","appos"}
ACTOR_DEPS      = {"nsubj","csubj","agent"}
SUBJ_DEPS       = {"nsubj","nsubjpass","csubj","csubjpass","expl"}
SHARED_AUX_DEPS = {"aux","auxpass","neg"}
COMPLEMENT_DEPS = {"prep","dobj","attr","acomp","xcomp","ccomp","oprd","dative","agent","dep","pobj"}
STOP_DEPS       = {"advcl","conj","cc","relcl","parataxis","mark","punct"}
CONTENT_POS     = ("VERB","NOUN","ADJ","AUX","ADV")
LEMMA_FIX = {"coadministere": "coadminister", "co-administere": "co-administer", "administere": "administer"}


def lemma(tok):
    l = tok.lemma_.lower()
    return LEMMA_FIX.get(l, l)


def sentence_case(s):
    s = re.sub(r"\s+([,.;:)])", r"\1", s.strip())
    s = re.sub(r"\(\s+", "(", s)
    return s[:1].upper() + s[1:] + ("" if s.endswith(".") else ".")


def surface(toks):
    # Clause text with the original spacing: "warfarin's", "half-life" (word-internal - and / are put back),
    # and a single space wherever tokens were cut out.
    ids = {t.i for t in toks}
    doc = toks[0].doc
    for t in toks[:-1]:
        nxt = t.i + 1
        if (nxt + 1 in ids and nxt not in ids and doc[nxt].text in ("-", "/")
                and not t.whitespace_ and not doc[nxt].whitespace_):
            ids.add(nxt)
    out, prev = [], None
    for i in sorted(ids):
        if prev is not None:
            out.append(doc[prev].whitespace_ if i == prev + 1 else " ")
        out.append(doc[i].text)
        prev = i
    return sentence_case("".join(out))


# ── drug mentions ─────────────────────────────────────────────────────────────────────────────
CLASS_NOUNS = ("drugs", "drug", "agents", "agent", "medications", "medication", "medicines", "medicine")


def name_variants(name):
    # Surface variants of a LABELLED drug name: singular/plural of the last word and class-noun swaps
    # ('cns depressant drugs' -> '... medications' / '... agents'). Only used for the labelled pair.
    name = name.lower().strip()
    variants = {name}
    words = name.split()
    if not words:
        return variants
    head, last = words[:-1], words[-1]
    if len(last) >= 5:
        variants.add(" ".join(head + [last[:-1] if last.endswith("s") and not last.endswith("ss") else last + "s"]))
    if last in CLASS_NOUNS and head:
        variants.update(" ".join(head + [c]) for c in CLASS_NOUNS)
    return {v for v in variants if len(v) > 2}


def _labelled(name, known_pair):
    # `name` (lower-case span text) IS one of the labelled drugs: exact / variant match, or a longer span that
    # contains it ('albendazole sulfoxide', 'r-warfarin'), or a short form of it ('cyclosporine' vs 'cyclosporine a').
    # Generic words never qualify through the 'short form' route.
    for k in known_pair:
        vs = name_variants(k)
        if name in vs:
            return True
        if any(len(v) >= 4 and v in name for v in vs):
            return True
        if len(name) >= 5 and name not in GENERIC_STOP and any(name in v for v in vs):
            return True
    return False


def canon(name, known_pair):
    # Which labelled drug a span refers to, so 'cyclosporine' and 'cyclosporine a' count as ONE drug.
    # An exact / variant match wins; otherwise the longest labelled name that matches.
    ordered = sorted(known_pair, key=len, reverse=True)
    for k in ordered:
        if name in name_variants(k):
            return k
    for k in ordered:
        if _labelled(name, {k}):
            return k
    return name


def entity_spans(sent_doc, known_pair):
    pair_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    pair_matcher.add("KNOWN_PAIR", [nlp.make_doc(v) for n in known_pair for v in name_variants(n)])
    spans = [sent_doc[s:e] for _, s, e in list(global_matcher(sent_doc)) + list(pair_matcher(sent_doc))]
    first, mentions = [], defaultdict(list)
    for sp in filter_spans(spans):
        name = sp.text.lower()
        labelled = _labelled(name, known_pair)
        if not labelled:
            if name in GENERIC_STOP or len(name) <= 2 or sp.root.pos_ not in {"NOUN", "PROPN", "X"}:
                continue
            if sp.end < len(sp.doc) and sp.doc[sp.end].lower_ in DESCRIPTOR_NOUNS:
                continue        # "the antibacterial action of ...": a kind of effect, not a drug
        key = canon(name, known_pair) if labelled else name
        if key not in mentions:
            first.append((sp, key))
        mentions[key].append(sp)
    return first, mentions


# ── predicate and clause ──────────────────────────────────────────────────────────────────────
def modal_advice(t):
    # 'should not be COMBINED', 'must be GIVEN with food' (see COADMIN_VERBS)
    return lemma(t) in COADMIN_VERBS and any(c.dep_ == "aux" and c.text.lower() in ADVICE_MODALS for c in t.children)


def is_trigger(t):
    return t.pos_ in CONTENT_POS and (lemma(t) in STRONG or modal_advice(t))


def trigger_name(t):
    return lemma(t) if lemma(t) in STRONG else lemma(t) + "+modal"


def supports(p):
    # P governs a checkable claim: an effect / advice / no-change word occurs in its main clause.
    return any(is_trigger(t) and t.dep_ != "advcl" for t in p.subtree)


def find_predicate(a, b):
    # Nearest ancestor of drug `a` that also governs drug `b` and carries an effect/advice predicate.
    # 'taken' / 'combined' are skipped automatically (they are not in the lexicon) and the search climbs to
    # the real predicate: 'Rifampin taken with diltiazem DECREASES the plasma concentration of X'.
    cur, seen = a.root, set()
    while True:
        if cur.head.i == cur.i:
            return None
        cur = cur.head
        if cur.i in seen:
            return None
        seen.add(cur.i)
        if b.root.i in {t.i for t in cur.subtree} and cur.pos_ in ("VERB", "AUX", "ADJ") and supports(cur):
            return cur


def is_condition(advcl):
    # 'if renal function is reduced', 'when given at high doses' - a condition, not a separate claim.
    return any(ch.dep_ in ("mark", "advmod") and ch.text.lower() in CONDITION_MARKS for ch in advcl.children)


def core_clause(p, needed_idx, keep_context=False):
    # The predicate plus everything it needs. Removed: coordinated predicates and relative clauses (separate
    # claims), reason / purpose clauses, and - unless keep_context - context phrases ('in patients', 'during
    # treatment') and condition clauses ('if ...') that contain neither drug.
    # The prepositional object of a light verb ('results IN ...', 'leads TO ...') is never removed.
    light = lemma(p) in LIGHT
    drop, conj_kept = set(), False
    for c in p.children:
        has_need = any(t.i in needed_idx for t in c.subtree)
        if c.dep_ == "punct":
            drop.update(t.i for t in c.subtree)
        elif has_need:
            conj_kept |= c.dep_ == "conj"
            continue
        elif c.dep_ == "advcl":
            if not (keep_context and is_condition(c)):
                drop.update(t.i for t in c.subtree)
        elif c.dep_ == "prep" and c.text.lower() in CONTEXT_PREPS:
            if not (keep_context or light):
                drop.update(t.i for t in c.subtree)
        elif c.dep_ in ("conj", "cc", "parataxis", "relcl"):
            drop.update(t.i for t in c.subtree)
        elif c.dep_ not in KEEP_DEPS:
            drop.update(t.i for t in c.subtree)
    if conj_kept:                     # a conjunct holds a drug, so keep its 'and' too (grammatical text)
        drop -= {c.i for c in p.children if c.dep_ in ("cc", "preconj")}
    return [t for t in p.subtree if t.i not in drop and t.pos_ != "PUNCT"]


def conjunct_clause(p, c, needed_idx, keep_context=False, full=False):
    # Atom for a coordinated predicate c of p: 'Coadministration of X and Y increases A and MAY REDUCE B'
    # -> 'Coadministration of X and Y may reduce B'. c borrows p's subject (and p's modal / negation when c has
    # no tense of its own), and drops its own further conjuncts.
    doc = c.doc
    if full:
        chain = {t.i for ch in c.children if ch.dep_ in ("conj", "cc", "preconj", "punct") for t in ch.subtree}
        ids = {t.i for t in c.subtree} - chain
    else:
        ids = {t.i for t in core_clause(c, needed_idx, keep_context)}
    if not any(ch.dep_ in SUBJ_DEPS for ch in c.children):
        for ch in p.children:
            if ch.dep_ in SUBJ_DEPS:
                ids |= {t.i for t in ch.subtree}
        if not is_finite(c):
            ids |= {ch.i for ch in p.children if ch.dep_ in SHARED_AUX_DEPS and ch.i < p.i}
    return [doc[i] for i in sorted(ids) if doc[i].pos_ != "PUNCT"]


def is_fragment(p, toks):
    # True when core_clause cut a COMPLEMENT off the end of the clause, e.g. 'are exclusively linked' (to renal
    # toxicity). Trimming a context adjunct ('in patients', 'during treatment'), a trailing clause or a conjunct
    # is intended and is not a fragment.
    last = toks[-1]
    if last.pos_ not in ("VERB", "AUX", "ADJ", "ADV"):
        return False
    kept = {t.i for t in toks}
    rest = [t for t in p.subtree if t.i > last.i and t.i not in kept and t.pos_ != "PUNCT"]
    if not rest:
        return False
    t = rest[0]
    return not (t.text.lower() in CONTEXT_PREPS or t.pos_ in ("SCONJ", "CCONJ")
                or t.dep_ in STOP_DEPS or t.tag_.startswith("W"))


def is_finite(p):
    FIN = {"VBZ", "VBP", "VBD", "MD"}
    if p.tag_ in FIN or any(c.dep_ in ("aux", "auxpass") and c.tag_ in FIN for c in p.children):
        return True
    # 'X and Y interact' sometimes gets the base-form tag VB instead of VBP. A main verb with its own subject
    # and no auxiliary (so not 'to interact') is a present-tense verb.
    return (p.tag_ == "VB" and p.dep_ in ("ROOT", "conj")
            and any(c.dep_ in SUBJ_DEPS for c in p.children)
            and not any(c.dep_ in ("aux", "auxpass") for c in p.children))


def triggers(toks, drug_idx):
    # The effect / advice / no-change words of a clause (drug names excluded).
    return [t for t in toks if is_trigger(t) and t.i not in drug_idx and t.text.lower() not in VOCAB]


def missing_complement(toks, drug_idx):
    # 'X combined with Y leads.' - the only effect words are light ones and none kept what it points to.
    trig = triggers(toks, drug_idx)
    if not trig or any(lemma(t) not in LIGHT for t in trig):
        return False
    ids = {t.i for t in toks}
    return not any(ch.i in ids and ch.dep_ in COMPLEMENT_DEPS for t in trig for ch in t.children)


def dangling(toks):
    # 'may add to.' / 'with X and Y'
    return (toks[0].pos_ in ("ADP", "CCONJ", "SCONJ", "PART")
            or toks[-1].pos_ in ("ADP", "AUX", "CCONJ", "SCONJ", "PART", "DET"))


def check_atom(pred, toks, full, has_pair, drug_idx):
    # All atom rules for one candidate clause. Returns (tokens, repaired) or None.
    if not toks or not has_pair(toks):                         # trimming must never lose a drug
        return None
    repaired = False
    if is_fragment(pred, toks) or missing_complement(toks, drug_idx):
        # trimming cut off a complement -> use the whole clause; a fragment is worse than no atom
        if has_pair(full) and len(full) <= FALLBACK_MAX_LEN and [t.i for t in full] != [t.i for t in toks]:
            toks, repaired = full, True
        else:
            return None
    if not (MIN_LEN <= len(toks) <= (FALLBACK_MAX_LEN if repaired else MAX_LEN)):
        return None
    if dangling(toks):
        return None
    if not triggers(toks, drug_idx):                           # co-administration only -> not a claim
        return None
    if missing_complement(toks, drug_idx):                     # light verb with nothing after it, even in full
        return None
    return toks, repaired


def pick_version(pred, keep_toks, trim_toks, full, has_pair, drug_idx):
    # Prefer the clause WITH its context phrases; fall back to the trimmed clause (e.g. when too long).
    # context_trimmed = some context phrase present in the full-context version is missing from the atom.
    tries = [keep_toks] if KEEP_CONTEXT_PHRASES else []
    if not tries or [t.i for t in trim_toks] != [t.i for t in keep_toks]:
        tries.append(trim_toks)
    for toks in tries:
        r = check_atom(pred, toks, full, has_pair, drug_idx)
        if r is not None:
            final, repaired = r
            cut = {t.i for t in keep_toks} - {t.i for t in final}
            return final, repaired, bool(cut)
    return None


def role_of(span, preds):
    for pr in preds:
        if pr is None:
            continue
        for ch in pr.children:
            if any(t.i == span.root.i for t in ch.subtree):
                return "actor" if ch.dep_ in ACTOR_DEPS else "patient"
    return "patient"


def _first_inside(ids, spans):
    return next((sp for sp in spans if {t.i for t in sp} <= ids), None)


def make_atom(toks, pred, shared, ka, kb, mentions, tier, repaired, trimmed, drug_idx):
    ids = {t.i for t in toks}
    a, b = _first_inside(ids, mentions[ka]), _first_inside(ids, mentions[kb])
    ra, rb = role_of(a, [pred, shared]), role_of(b, [pred, shared])
    if ra == "actor" and rb != "actor":
        sub, obj, role_src = a, b, "dependency"
    elif rb == "actor" and ra != "actor":
        sub, obj, role_src = b, a, "dependency"
    else:                                                      # both / neither are actors -> text order
        (sub, obj), role_src = ((a, b) if a.start < b.start else (b, a)), "text_order"
    pair_ids = {t.i for t in a} | {t.i for t in b}
    rel_toks = [t for t in toks if t.i not in pair_ids]
    edge = r"^(and|or|with|of|by|to)\s+|\s+(and|or|with|of|by|to)$"
    rel = re.sub(edge, "", " ".join(t.text.lower() for t in rel_toks)).strip()
    rel_lemma = re.sub(edge, "", " ".join(lemma(t) for t in rel_toks)).strip()
    others = sorted(k for k, sps in mentions.items() if k not in (ka, kb) and _first_inside(ids, sps) is not None)
    return {
        "sub_extract": sub.text.lower(), "rel_extract": rel, "rel_lemma": rel_lemma, "obj_extract": obj.text.lower(),
        "role_source": role_src, "hypothesis": surface(toks),
        "source_tier": tier, "repaired": repaired, "context_trimmed": trimmed,
        "trigger_words": ";".join(sorted({trigger_name(t) for t in triggers(toks, drug_idx)})),
        "other_drugs": ";".join(others),
    }


def clause_atoms(sent_doc, first, mentions, restrict, known_pair):
    # Every atom of one sentence: one per drug pair (+ one per coordinated predicate when SPLIT_CONJUNCTS).
    # restrict = True -> only the labelled pair may form an atom.
    out = []
    for i, (a, ka) in enumerate(first):
        for b, kb in first[i + 1:]:
            if ka == kb:
                continue
            if restrict and not (ka in known_pair and kb in known_pair):
                continue
            p = find_predicate(a, b) or find_predicate(b, a)
            if p is None or not is_finite(p):
                continue
            need = {t.i for t in a} | {t.i for t in b}
            need_all = {t.i for sp in mentions[ka] + mentions[kb] for t in sp}

            def has_pair(toks, ka=ka, kb=kb):
                ids = {t.i for t in toks}
                return _first_inside(ids, mentions[ka]) is not None and _first_inside(ids, mentions[kb]) is not None

            cands = [("clause_predicate", p, None,
                      core_clause(p, need, True), core_clause(p, need, False),
                      [t for t in p.subtree if t.pos_ != "PUNCT"])]
            if SPLIT_CONJUNCTS and any(c.dep_ == "conj" and any(t.i in need for t in c.subtree) for c in p.children):
                # 'X is metabolized by the liver and increases Y levels': the drug pair only meets across two
                # predicates, so the conjunct below gives the real atom ('X increases Y levels').
                cands = []
            if SPLIT_CONJUNCTS:
                for c in p.conjuncts:
                    if c.pos_ in ("VERB", "AUX", "ADJ"):
                        cands.append(("conjunct_predicate", c, p,
                                      conjunct_clause(p, c, need_all, True), conjunct_clause(p, c, need_all, False),
                                      conjunct_clause(p, c, need_all, full=True)))
            for tier, pred, shared, keep_toks, trim_toks, full in cands:
                r = pick_version(pred, keep_toks, trim_toks, full, has_pair, need_all)
                if r is None:
                    continue
                toks, repaired, trimmed = r
                out.append(make_atom(toks, pred, shared, ka, kb, mentions, tier, repaired, trimmed, need_all))
    return out


print("Extraction helpers defined.")

Extraction helpers defined.


## atomize_dataframe

**Input from notebook 1.** One row per response, with `original_id`, `premise`, `e1_text`, `e2_text`, `hypothesis` (the response text) and `label` (the response-level label).

In [8]:
ATOM_FIELDS = ["sub_extract", "rel_extract", "rel_lemma", "obj_extract", "role_source", "hypothesis",
               "source_tier", "repaired", "context_trimmed", "trigger_words", "other_drugs"]
OUT_COLUMNS = ["source_row", "original_id", "Entity1", "Entity2", "premise", "source_text", "response_label",
               "claim_idx", "claim_text", "label", "label_source"] + ATOM_FIELDS + ["filter_reason"]


def parse_claims(value):
    # Per-claim labels from notebook 1. Returns [(text, label), ...], or None for an empty cell.
    # Raises ValueError when the cell cannot be read.
    if isinstance(value, list):
        items = value
    elif value is None or pd.isna(value) or not str(value).strip():
        return None
    else:
        s = str(value).strip()
        try:
            items = json.loads(s)
        except ValueError:
            try:
                items = ast.literal_eval(s)
            except (ValueError, SyntaxError) as e:
                raise ValueError(f"unreadable claims cell: {s[:80]!r}") from e
    if not isinstance(items, list):
        raise ValueError("claims cell is not a list")
    out = []
    for it in items:
        if not isinstance(it, dict) or "text" not in it or "label" not in it:
            raise ValueError(f"each claim needs 'text' and 'label': {it!r}")
        out.append((str(it["text"]).strip(), str(it["label"]).strip().lower()))
    return out or None


def _no_atom(info, reason):
    return {**info, **dict.fromkeys(ATOM_FIELDS), "source_tier": "filtered_out", "filter_reason": reason}


def atomize_dataframe(df, *, id_col="original_id", text_col="hypothesis", premise_col="premise",
                      e1_col="e1_text", e2_col="e2_text", label_col="label", claims_col="claims", split_name=""):
    has_claims = claims_col in df.columns
    if not has_claims:
        print(f"[{split_name}] WARNING: no '{claims_col}' column -> every atom inherits the label of its whole "
              f"response (old behaviour). Add per-claim labels in notebook 1.")
    records, units, stats = [], [], Counter()

    for pos, (_, row) in enumerate(df.iterrows()):
        src  = int(row["source_row"]) if "source_row" in df.columns else pos
        text = "" if pd.isna(row[text_col]) else str(row[text_col]).strip()
        e1   = str(row[e1_col]).strip() if pd.notna(row[e1_col]) else ""
        e2   = str(row[e2_col]).strip() if pd.notna(row[e2_col]) else ""
        resp_label = str(row[label_col]).strip().lower()
        base = {"source_row": src, "original_id": row[id_col], "Entity1": e1, "Entity2": e2,
                "premise": row[premise_col], "source_text": text, "response_label": resp_label}
        whole = {**base, "claim_idx": -1, "claim_text": None, "label": resp_label, "label_source": "response"}

        if not text:
            records.append(_no_atom(whole, "empty_text"))
            continue
        if not e1 or not e2 or e1.lower() == e2.lower():
            records.append(_no_atom(whole, "same_or_missing_drug"))
            continue

        claims = None
        if has_claims:
            try:
                claims = parse_claims(row[claims_col])
            except ValueError:
                stats["unreadable claims cell"] += 1
                records.append(_no_atom(whole, "unreadable_claims"))
                continue
            if claims is None:
                stats["empty claims cell -> response label used"] += 1
            elif " ".join(" ".join(t for t, _ in claims).split()) != " ".join(text.split()):
                stats["claims text differs from hypothesis"] += 1
        if claims:
            for k, (ctext, clabel) in enumerate(claims):
                units.append(({**base, "claim_idx": k, "claim_text": ctext, "label": clabel,
                               "label_source": "claim"}, ctext))
        else:
            units.append((whole, text))

    seen = {}
    for (info, utext), doc in zip(units, nlp.pipe((u[1] for u in units), batch_size=256)):
        lab = info["label"]
        if lab not in LABELS:
            records.append(_no_atom(info, "bad_label"))
            continue
        known_pair = {info["Entity1"].lower(), info["Entity2"].lower()}
        restrict = NEUTRAL_SAME_PAIR or lab != "neutral"
        n_before = len(records)
        for sentence in doc.sents:
            sent_doc = sentence.as_doc()
            first, mentions = entity_spans(sent_doc, known_pair)
            if len(first) < 2:
                continue
            for atom in clause_atoms(sent_doc, first, mentions, restrict, known_pair):
                key = (info["source_row"], atom["hypothesis"])
                if key in seen:
                    stats["same atom twice in one response" + (" (different labels!)" if seen[key] != lab else "")] += 1
                    continue
                seen[key] = lab
                records.append({**info, **atom, "filter_reason": None})
        if len(records) == n_before:
            records.append(_no_atom(info, "no_atom"))

    out = pd.DataFrame(records, columns=OUT_COLUMNS)
    out = out.sort_values(["source_row", "claim_idx"], kind="stable").reset_index(drop=True)
    out.attrs["stats"] = dict(stats)
    return out


def _pct(s):
    return ", ".join(f"{k} {v:.1%}" for k, v in s.items())


def report_coverage(df_atomic, split_name):
    is_atom = df_atomic["source_tier"] != "filtered_out"
    ok = df_atomic[is_atom]
    rows = df_atomic.assign(_ok=is_atom).groupby("source_row").agg(
        response_label=("response_label", "first"), covered=("_ok", "any"))
    n_cov = int(rows["covered"].sum())
    print(f"[{split_name}] responses with >=1 atom: {n_cov:,}/{len(rows):,} ({rows['covered'].mean():.1%})")
    print("  by response label :", _pct(rows.groupby("response_label")["covered"].mean()))
    by_claim = df_atomic["label_source"] == "claim"
    if by_claim.any():
        claims = df_atomic[by_claim].assign(_ok=is_atom[by_claim]).groupby(["source_row", "claim_idx"]).agg(
            label=("label", "first"), covered=("_ok", "any"))
        print(f"  claims with >=1 atom: {int(claims['covered'].sum()):,}/{len(claims):,} ({claims['covered'].mean():.1%})")
        print("  by claim label    :", _pct(claims.groupby("label")["covered"].mean()))
    print(f"  atoms: {len(ok):,}  (avg {len(ok) / max(n_cov, 1):.2f} per covered response)")
    print("  atom labels       :", ok["label"].value_counts().to_dict())
    print(f"  from split conjuncts: {(ok['source_tier'] == 'conjunct_predicate').sum():,} | "
          f"repaired with whole clause: {int(ok['repaired'].sum()):,} | "
          f"context trimmed: {int(ok['context_trimmed'].sum()):,} | "
          f"roles by text order: {(ok['role_source'] == 'text_order').sum():,}")
    print("  no atom, by reason:", df_atomic.loc[~is_atom, "filter_reason"].value_counts().to_dict())
    if df_atomic.attrs.get("stats"):
        print("  notes             :", df_atomic.attrs["stats"])
    print()


print("atomize_dataframe() and report_coverage() defined.")

atomize_dataframe() and report_coverage() defined.


## Load holistic responses

Rows left over from the removed fake-drug scenario are dropped here: `label` or `scenario` equal to `fake_drug`, or a true `fake_drug` flag. The old columns `fake_drug`, `entity_valid` and `swap_drug` are removed. `source_row` is the row number after this cleaning, and every atom keeps it.

In [10]:
LEGACY_COLS = ["fake_drug", "entity_valid", "swap_drug"]


def _truthy(v):
    return v is True or str(v).strip().lower() in ("true", "1", "yes")


def load_holistic(path, name):
    df = pd.read_csv(path)
    fake = df["label"].astype(str).str.strip().str.lower().eq("fake_drug")
    if "scenario" in df.columns:
        fake |= df["scenario"].astype(str).str.strip().str.lower().eq("fake_drug")
    if "fake_drug" in df.columns:
        fake |= df["fake_drug"].map(_truthy)
    if fake.any():
        print(f"[{name}] dropped {int(fake.sum()):,} legacy fake_drug rows")
    df = df.loc[~fake].drop(columns=[c for c in LEGACY_COLS if c in df.columns]).reset_index(drop=True)
    df.insert(0, "source_row", range(len(df)))
    print(f"[{name}] {len(df):,} responses | labels: {df['label'].value_counts().to_dict()}")
    return df


holistic = {name: load_holistic(f"data/prompt_test/prompt_test_normal_20261028_153352.csv", name) for name in ("train", "val", "test")}

[train] 3 responses | labels: {'entailment': 1, 'contradiction': 1, 'neutral': 1}
[val] 3 responses | labels: {'entailment': 1, 'contradiction': 1, 'neutral': 1}
[test] 3 responses | labels: {'entailment': 1, 'contradiction': 1, 'neutral': 1}


## Run atomisation

In [11]:
atomic_raw = {}
for name, df in holistic.items():
    atomic_raw[name] = atomize_dataframe(df, split_name=name)
    report_coverage(atomic_raw[name], name)

[train] responses with >=1 atom: 3/3 (100.0%)
  by response label : contradiction 100.0%, entailment 100.0%, neutral 100.0%
  claims with >=1 atom: 11/12 (91.7%)
  by claim label    : contradiction 100.0%, entailment 87.5%, neutral 100.0%
  atoms: 11  (avg 3.67 per covered response)
  atom labels       : {'entailment': 7, 'contradiction': 2, 'neutral': 2}
  from split conjuncts: 0 | repaired with whole clause: 0 | context trimmed: 0 | roles by text order: 5
  no atom, by reason: {'no_atom': 1}

[val] responses with >=1 atom: 3/3 (100.0%)
  by response label : contradiction 100.0%, entailment 100.0%, neutral 100.0%
  claims with >=1 atom: 11/12 (91.7%)
  by claim label    : contradiction 100.0%, entailment 87.5%, neutral 100.0%
  atoms: 11  (avg 3.67 per covered response)
  atom labels       : {'entailment': 7, 'contradiction': 2, 'neutral': 2}
  from split conjuncts: 0 | repaired with whole clause: 0 | context trimmed: 0 | roles by text order: 5
  no atom, by reason: {'no_atom': 1}

[t

## Save

- `data/atomic_*.csv` holds atoms only. Responses with no atom are left out. The diagnostic columns `repaired`, `context_trimmed`, `trigger_words`, `other_drugs` and `filter_reason` are not saved (`DROP_FROM_CSV`).
- `data/holistic_*_clean.csv` holds every response after the fake-drug cleaning, with `n_atoms` and `covered` added.

In notebook 4, evaluate the holistic models twice: on all rows, and on `covered == True`. The second result is the one to compare with the atomic models, because both then use the same responses.

In [ ]:
os.makedirs("data", exist_ok=True)
# Kept in memory (coverage report, lexicon audit, notebook 1's atomiser gate) but not written to the CSV
DROP_FROM_CSV = ["repaired", "context_trimmed", "trigger_words", "other_drugs", "filter_reason"]
atomic = {}
for name, raw in atomic_raw.items():
    atoms = raw[raw["source_tier"] != "filtered_out"].reset_index(drop=True)
    atomic[name] = atoms
    atoms.drop(columns=DROP_FROM_CSV).to_csv(f"data/atomic_{name}.csv", index=False)

    clean = holistic[name].copy()
    clean["n_atoms"] = clean["source_row"].map(atoms.groupby("source_row").size()).fillna(0).astype(int)
    clean["covered"] = clean["n_atoms"] > 0
    clean.to_csv(f"data/holistic_{name}_clean.csv", index=False)

    print(f"Saved data/atomic_{name}.csv ({len(atoms):,} atoms) and "
          f"data/holistic_{name}_clean.csv ({len(clean):,} responses, {int(clean['covered'].sum()):,} covered)")
    print("  atom labels:", atoms["label"].value_counts().to_dict())

## Check: sample atoms (val)

The sample is taken from val, not test, so that adjusting the rules never depends on looking at test output.

In [13]:
print("Most common relations (train):")
print(atomic["train"]["rel_extract"].value_counts().head(20).to_string())
print()
cols = ["Entity1", "Entity2", "label", "response_label", "sub_extract", "rel_extract", "obj_extract",
        "hypothesis", "source_tier", "context_trimmed"]
atomic["val"][cols].head(20)

Most common relations (train):
rel_extract
patients who are on therapy may require downward adjustment of their anticoagulant dosage when taking    2
have been shown to depress plasma prothrombin activity in patients taking                                2
patients on therapy may require downward adjustment of their anticoagulant dosage when taking            1
a downward adjustment of dosage may be required when taking                                              1
dosage may need a downward adjustment in patients receiving                                              1
therapy shows no difference in dosage adjustment when combined                                           1
increase plasma prothrombin activity during therapy                                                      1
interfere when administered concurrently                                                                 1
require a downward dosage adjustment of twenty percent                                               

,Entity1,Entity2,label,response_label,sub_extract,rel_extract,obj_extract,hypothesis,source_tier,context_trimmed
0,tetracyclines,anticoagulant,entailment,entailment,anticoagulant,patients on therapy may require downward adjus...,tetracyclines,Patients on anticoagulant therapy may require ...,clause_predicate,False
1,tetracyclines,anticoagulant,entailment,entailment,anticoagulant,a downward adjustment of dosage may be require...,tetracyclines,A downward adjustment of anticoagulant dosage ...,clause_predicate,False
2,tetracyclines,anticoagulant,entailment,entailment,anticoagulant,dosage may need a downward adjustment in patie...,tetracyclines,Anticoagulant dosage may need a downward adjus...,clause_predicate,False
3,tetracyclines,anticoagulant,entailment,contradiction,anticoagulant,patients who are on therapy may require downwa...,tetracyclines,Patients who are on anticoagulant therapy may ...,clause_predicate,False
4,tetracyclines,anticoagulant,entailment,contradiction,tetracyclines,have been shown to depress plasma prothrombin ...,anticoagulant,Tetracyclines have been shown to depress plasm...,clause_predicate,False
5,tetracyclines,anticoagulant,contradiction,contradiction,anticoagulant,therapy shows no difference in dosage adjustme...,tetracyclines,Anticoagulant therapy shows no difference in d...,clause_predicate,False
6,tetracyclines,anticoagulant,contradiction,contradiction,tetracyclines,increase plasma prothrombin activity during th...,anticoagulant,Tetracyclines increase plasma prothrombin acti...,clause_predicate,False
7,tetracyclines,anticoagulant,entailment,neutral,anticoagulant,patients who are on therapy may require downwa...,tetracyclines,Patients who are on anticoagulant therapy may ...,clause_predicate,False
8,tetracyclines,anticoagulant,entailment,neutral,tetracyclines,have been shown to depress plasma prothrombin ...,anticoagulant,Tetracyclines have been shown to depress plasm...,clause_predicate,False
9,tetracyclines,anticoagulant,neutral,neutral,tetracyclines,interfere when administered concurrently,anticoagulant,Tetracyclines and anticoagulant interfere when...,clause_predicate,False


## Lexicon audit

For each word added to the lexicon after reading generated text, this counts the atoms that contain it in each split. *Depends only on added words* counts atoms that would disappear if the additions were removed.

The audit cannot show *where* a word was found. If any word was chosen by reading test hypotheses, say so in the thesis, or keep only words you can justify from train/val.

In [14]:
def lexicon_audit(atomic):
    words = {name: atoms["trigger_words"].fillna("").map(lambda s: set(filter(None, s.split(";"))))
             for name, atoms in atomic.items()}
    tab = pd.DataFrame({name: {w: int(ws.map(lambda s: w in s).sum()) for w in sorted(LEXICON_ADDED)}
                        for name, ws in words.items()})
    others = [c for c in tab.columns if c != "test"]
    tab["only_in_test"] = (tab["test"] > 0) & (tab[others].sum(axis=1) == 0)
    print(tab.to_string())
    print()
    for name, ws in words.items():
        dep = ws.map(lambda s: bool(s) and s <= LEXICON_ADDED)
        print(f"[{name}] atoms that depend only on added words: {int(dep.sum()):,}/{len(ws):,} ({dep.mean():.1%})")


lexicon_audit(atomic)

            train  val  test  only_in_test
advise          0    0     0         False
comparable      0    0     0         False
constant        0    0     0         False
cut             0    0     0         False
difference      1    1     1         False
encourage       0    0     0         False
endorse         0    0     0         False
equivalent      0    0     0         False
finding         0    0     0         False
halve           0    0     0         False
instruct        0    0     0         False
maintain        0    0     0         False
mandate         0    0     0         False
need            1    1     1         False
remain          0    0     0         False
same            0    0     0         False
stay            0    0     0         False
support         0    0     0         False
target          0    0     0         False
urge            0    0     0         False

[train] atoms that depend only on added words: 2/11 (18.2%)
[val] atoms that depend only on adde